In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Ejercicio Parte 4").getOrCreate()
sc = spark.sparkContext

# Ejercicios Prácticos: Acumuladores, Variables Broadcast y Persistencia

**Contexto:** Los datos adjuntos como recurso a esta lección contienen los importes de las ventas de un día en un supermercado.

---

### **1. Creación de RDD y uso de Acumuladores (Inciso A)**
Cree un RDD llamado `importes` a partir de los datos adjuntos a esta lección. Emplee **acumuladores** para obtener de manera eficiente:
* El número total de ventas realizadas.
* El importe total consolidado de todas las ventas.

### **2. Estrategia con Variables Broadcast**
Si se conoce que a cada venta hay que restarle un importe fijo igual a **10 pesos** por temas de impuestos:
* ¿Cómo restaría este impuesto de cada venta utilizando una **variable broadcast** para acelerar el proceso de distribución?

### **3. Creación del RDD Filtrado (Inciso B)**
Cree un RDD llamado `ventas_sin_impuestos` a partir de la propuesta del **Inciso A**, aplicando la lógica de la variable broadcast para que contenga los importes finales con el impuesto ya restado.

### **4. Destrucción de la Variable**
Destruya de forma definitiva la variable broadcast creada, inmediatamente después de haberla empleado para generar el RDD del **Inciso B**.

### **5. Pruebas de Persistencia**
Persista el RDD `ventas_sin_impuestos` configurándolo secuencialmente en los siguientes niveles de persistencia:
* Memoria solamente (`MEMORY_ONLY`).
* Disco solamente (`DISK_ONLY`).
* Memoria y disco (`MEMORY_AND_DISK`).

In [4]:
importes = sc.textFile("./data/")

In [5]:
importes.take(10)

['527', '386', '701', '240', '941', '27', '396', '56', '456', '148']

In [6]:
total_ventas = sc.accumulator(0)

In [7]:
importe_total = sc.accumulator(0)

In [8]:
importes.foreach(lambda x: total_ventas.add(1))

In [9]:
importes.foreach(lambda x: importe_total.add(float(x)))

In [10]:
total_ventas.value

10000

In [11]:
importe_total.value

5042335.0

In [13]:
# hacerlo sin acumuladores

importes.count()

10000

In [15]:
impuesto = sc.broadcast(10)

In [17]:
# se transforman en floar porque los valores son strings

ventas_sin_impuestos = importes.map(lambda x: float(x) - impuesto.value)

In [18]:
print(ventas_sin_impuestos.take(10))

[517.0, 376.0, 691.0, 230.0, 931.0, 17.0, 386.0, 46.0, 446.0, 138.0]


In [19]:
impuesto.destroy()

In [21]:
from pyspark.storagelevel import StorageLevel

In [23]:
# opcion mas sencilla para persistirlo en memoria

ventas_sin_impuestos.cache()

PythonRDD[8] at RDD at PythonRDD.scala:56

In [24]:
# para cambiar el nivel de persistencia debemos hacerle primeramente un unpersist

ventas_sin_impuestos.unpersist()

PythonRDD[8] at RDD at PythonRDD.scala:56

In [25]:
ventas_sin_impuestos.persist(StorageLevel.DISK_ONLY)

PythonRDD[8] at RDD at PythonRDD.scala:56

In [26]:
ventas_sin_impuestos.unpersist()

PythonRDD[8] at RDD at PythonRDD.scala:56

In [27]:
ventas_sin_impuestos.persist(StorageLevel.MEMORY_AND_DISK)

PythonRDD[8] at RDD at PythonRDD.scala:56

In [28]:
# Asi podemos ir variando los niveles de persistencia de los RDDs